# 11b — Validation Evaluation

This notebook evaluates the **tuned Week 5 models** on the held-out **validation datasets** and compares validation performance across dataset/model combinations.

## Objective

The notebook:

- locates and loads the tuned model artifacts saved during Week 5;
- loads the corresponding held-out validation datasets;
- evaluates every available dataset/model combination without re-fitting;
- calculates validation accuracy, balanced accuracy, macro F1, macro precision, and macro recall;
- generates and exports confusion matrices;
- creates validation comparison tables and figures;
- ranks the candidate models using **macro F1 as the primary selection metric** and **balanced accuracy as the tie-breaker**;
- identifies the strongest candidate model(s) to move forward.

## Required outputs

- `validation_results.csv`
- validation confusion matrices (`.csv` and `.png`)
- validation comparison tables
- updated validation figures
- validation metrics summary
- best-candidate justification

> **Important:** Validation data are used for evaluation only. No model fitting, preprocessing fitting, feature selection, or hyperparameter tuning is performed in this notebook.

## 1. Libraries and project paths

In [ ]:
from pathlib import Path
import sys
import warnings
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 200)

cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing the src directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
VALIDATION_DIR = DATA_DIR / "validation"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUTS_DIR / "metrics"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"

for directory in [METRICS_DIR, TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data: {PROCESSED_DIR}")
print(f"Validation data: {VALIDATION_DIR}")
print(f"Models: {MODELS_DIR}")
print(f"Outputs: {OUTPUTS_DIR}")

## 2. Validation configuration

In [ ]:
RANDOM_STATE = 42
TARGET_COLUMN = "label"
PARTICIPANT_ID_COLUMN = "patient_id"

EXCLUDED_COLUMNS = [
    PARTICIPANT_ID_COLUMN,
    TARGET_COLUMN,
    "condition_group",
    "condition_original",
]

# These are the integrated modeling datasets used by the project.
# The loader checks several common validation-file names for each dataset.
DATASETS = [
    "demographics_questionnaire",
    "wearable_questionnaire",
    "multimodal_full",
]

VALIDATION_FILE_CANDIDATES = {
    dataset: [
        VALIDATION_DIR / f"{dataset}_validation.csv",
        PROCESSED_DIR / f"{dataset}_validation.csv",
        TABLES_DIR / f"{dataset}_validation.csv",
        TABLES_DIR / f"{dataset}_selected_validation_dataset.csv",
    ]
    for dataset in DATASETS
}

# Search common locations used for saved tuned models.
MODEL_SEARCH_DIRS = [
    MODELS_DIR / "tuned",
    MODELS_DIR,
    OUTPUTS_DIR / "models",
    OUTPUTS_DIR / "tuned_models",
]

MODEL_EXTENSIONS = [".joblib", ".pkl", ".pickle"]

# Primary metric used to select the final candidate(s).
PRIMARY_SELECTION_METRIC = "macro_f1"
TIE_BREAKER_METRIC = "balanced_accuracy"

print(f"Primary selection metric: {PRIMARY_SELECTION_METRIC}")
print(f"Tie-breaker metric: {TIE_BREAKER_METRIC}")

The notebook searches common project locations automatically so it can work with the artifacts created during Week 5 without hard-coding one machine-specific path. If your Week 5 notebook used a different folder or filename convention, add that location to `MODEL_SEARCH_DIRS` or update the validation-file candidates above.

## 3. Helper — locate validation datasets

In [ ]:
def first_existing_path(paths):
    """Return the first existing path from a list, otherwise None."""
    for path in paths:
        if path.exists():
            return path
    return None


def locate_validation_datasets():
    """Locate one held-out validation dataset for each configured dataset."""
    records = []

    for dataset_name, candidates in VALIDATION_FILE_CANDIDATES.items():
        selected_path = first_existing_path(candidates)
        records.append({
            "dataset": dataset_name,
            "validation_file": str(selected_path) if selected_path else None,
            "status": "FOUND" if selected_path else "MISSING",
        })

    return pd.DataFrame(records)


validation_dataset_inventory = locate_validation_datasets()
display(validation_dataset_inventory)

## 4. Helper — discover tuned Week 5 model artifacts

In [ ]:
def discover_model_artifacts():
    """Discover saved tuned model files and map them to configured datasets."""
    records = []

    for search_dir in MODEL_SEARCH_DIRS:
        if not search_dir.exists():
            continue

        for extension in MODEL_EXTENSIONS:
            for model_path in search_dir.rglob(f"*{extension}"):
                stem_lower = model_path.stem.lower()

                matched_dataset = None
                for dataset_name in DATASETS:
                    if dataset_name.lower() in stem_lower:
                        matched_dataset = dataset_name
                        break

                if matched_dataset is None:
                    continue

                model_name = model_path.stem
                for token in [
                    matched_dataset,
                    "tuned",
                    "best",
                    "model",
                    "pipeline",
                    "week5",
                    "week_5",
                ]:
                    model_name = model_name.replace(token, "")
                    model_name = model_name.replace(token.upper(), "")

                model_name = model_name.strip("_- ") or model_path.stem

                records.append({
                    "dataset": matched_dataset,
                    "model": model_name,
                    "model_file": str(model_path),
                })

    inventory = pd.DataFrame(records)

    if inventory.empty:
        return pd.DataFrame(columns=["dataset", "model", "model_file"])

    return (
        inventory
        .drop_duplicates(subset=["dataset", "model_file"])
        .sort_values(["dataset", "model"])
        .reset_index(drop=True)
    )


model_inventory = discover_model_artifacts()
display(model_inventory)

if model_inventory.empty:
    print(
        "No tuned model artifacts were discovered automatically. "
        "Check MODEL_SEARCH_DIRS and the Week 5 model filenames."
    )

### Optional manual model map

If automatic discovery does not match your Week 5 filenames, enter the exact saved model paths below. Leave the dictionary empty when automatic discovery works.

In [ ]:
MANUAL_MODEL_MAP = {
    # Example:
    # ("multimodal_full", "logistic_regression"): MODELS_DIR / "tuned" / "multimodal_full_logistic_regression_tuned.joblib",
    # ("multimodal_full", "random_forest"): MODELS_DIR / "tuned" / "multimodal_full_random_forest_tuned.joblib",
}

if MANUAL_MODEL_MAP:
    manual_records = [
        {
            "dataset": dataset_name,
            "model": model_name,
            "model_file": str(model_path),
        }
        for (dataset_name, model_name), model_path in MANUAL_MODEL_MAP.items()
    ]

    model_inventory = pd.DataFrame(manual_records)
    display(model_inventory)

## 5. Validation dataset checks

In [ ]:
validation_data = {}
validation_structure_records = []

for _, row in validation_dataset_inventory.iterrows():
    dataset_name = row["dataset"]
    file_path = row["validation_file"]

    if not file_path:
        continue

    df = pd.read_csv(
        file_path,
        dtype={PARTICIPANT_ID_COLUMN: str},
    )

    required_columns = {TARGET_COLUMN}
    missing_required = required_columns.difference(df.columns)

    assert not missing_required, (
        f"{dataset_name}: missing required columns "
        f"{sorted(missing_required)}"
    )

    assert df[TARGET_COLUMN].notna().all(), (
        f"{dataset_name}: validation target contains missing values."
    )

    validation_data[dataset_name] = df

    validation_structure_records.append({
        "dataset": dataset_name,
        "rows": len(df),
        "columns": df.shape[1],
        "unique_participants": (
            df[PARTICIPANT_ID_COLUMN].nunique()
            if PARTICIPANT_ID_COLUMN in df.columns
            else np.nan
        ),
        "n_classes": df[TARGET_COLUMN].nunique(),
        "validation_file": file_path,
    })

validation_structure_summary = pd.DataFrame(validation_structure_records)
display(validation_structure_summary)

validation_structure_summary.to_csv(
    TABLES_DIR / "validation_dataset_structure_summary.csv",
    index=False,
)

## 6. Class distribution of validation datasets

In [ ]:
validation_class_distribution = []

for dataset_name, df in validation_data.items():
    counts = (
        df[TARGET_COLUMN]
        .value_counts()
        .sort_index()
    )

    for label, count in counts.items():
        validation_class_distribution.append({
            "dataset": dataset_name,
            "label": label,
            "count": int(count),
            "percentage": count / len(df) * 100,
        })

validation_class_distribution = pd.DataFrame(
    validation_class_distribution
)

display(validation_class_distribution)

validation_class_distribution.to_csv(
    TABLES_DIR / "validation_class_distribution.csv",
    index=False,
)

## 7. Model-loading helper

In [ ]:
def load_saved_model(model_path):
    """Load a Week 5 artifact and return its prediction-capable estimator."""
    artifact = joblib.load(model_path)

    # Support the common case where Week 5 saved a dictionary containing
    # the fitted estimator plus metadata.
    if isinstance(artifact, dict):
        for key in ["pipeline", "model", "estimator", "best_estimator"]:
            if key in artifact and hasattr(artifact[key], "predict"):
                return artifact[key], artifact

    if hasattr(artifact, "predict"):
        return artifact, None

    raise TypeError(
        f"Saved artifact at {model_path} does not expose a predict() method."
    )

## 8. Feature alignment helper

In [ ]:
def prepare_validation_features(model, df):
    """Prepare raw validation features in the column order expected by the model."""
    default_feature_columns = [
        column
        for column in df.columns
        if column not in EXCLUDED_COLUMNS
    ]

    X = df[default_feature_columns].copy()

    # Full sklearn pipelines fitted on DataFrames often retain feature_names_in_.
    if hasattr(model, "feature_names_in_"):
        expected = list(model.feature_names_in_)
        missing = [column for column in expected if column not in df.columns]

        if missing:
            raise ValueError(
                "Validation data are missing features expected by the saved "
                f"model: {missing[:10]}"
            )

        X = df[expected].copy()

    return X

This notebook assumes the Week 5 artifacts are **already fitted tuned models**, ideally saved as complete sklearn pipelines containing preprocessing and feature-selection steps. The notebook calls `predict()` only; it does not call `fit()` on validation data.

## 9. Evaluate all tuned models on held-out validation data

In [ ]:
validation_metric_records = []
validation_prediction_records = []
validation_error_records = []
confusion_matrix_records = []

for _, model_row in model_inventory.iterrows():
    dataset_name = model_row["dataset"]
    model_name = model_row["model"]
    model_path = Path(model_row["model_file"])

    if dataset_name not in validation_data:
        validation_error_records.append({
            "dataset": dataset_name,
            "model": model_name,
            "error": "Validation dataset not found.",
        })
        continue

    df = validation_data[dataset_name].copy()
    y_validation = df[TARGET_COLUMN].copy()

    try:
        model, artifact_metadata = load_saved_model(model_path)
        X_validation = prepare_validation_features(model, df)

        # Evaluation only — no fitting is performed here.
        validation_predictions = model.predict(X_validation)

        metrics_record = {
            "dataset": dataset_name,
            "model": model_name,
            "n_validation": len(y_validation),
            "accuracy": accuracy_score(
                y_validation,
                validation_predictions,
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_validation,
                validation_predictions,
            ),
            "macro_f1": f1_score(
                y_validation,
                validation_predictions,
                average="macro",
                zero_division=0,
            ),
            "precision_macro": precision_score(
                y_validation,
                validation_predictions,
                average="macro",
                zero_division=0,
            ),
            "recall_macro": recall_score(
                y_validation,
                validation_predictions,
                average="macro",
                zero_division=0,
            ),
            "model_file": str(model_path),
        }

        validation_metric_records.append(metrics_record)

        prediction_df = pd.DataFrame({
            "dataset": dataset_name,
            "model": model_name,
            "y_true": y_validation.reset_index(drop=True),
            "y_pred": pd.Series(validation_predictions),
        })

        if PARTICIPANT_ID_COLUMN in df.columns:
            prediction_df.insert(
                2,
                PARTICIPANT_ID_COLUMN,
                df[PARTICIPANT_ID_COLUMN].reset_index(drop=True),
            )

        validation_prediction_records.extend(
            prediction_df.to_dict("records")
        )

        labels = sorted(
            pd.Series(y_validation)
            .dropna()
            .unique()
            .tolist()
        )

        cm = confusion_matrix(
            y_validation,
            validation_predictions,
            labels=labels,
        )

        cm_df = pd.DataFrame(
            cm,
            index=[f"True_{label}" for label in labels],
            columns=[f"Pred_{label}" for label in labels],
        )

        safe_model_name = (
            str(model_name)
            .replace(" ", "_")
            .replace("/", "_")
            .replace("\\", "_")
        )

        cm_csv_path = (
            METRICS_DIR
            / f"{dataset_name}_{safe_model_name}_validation_confusion_matrix.csv"
        )
        cm_df.to_csv(cm_csv_path)

        fig, ax = plt.subplots(figsize=(6, 5))
        image = ax.imshow(cm)
        fig.colorbar(image, ax=ax)

        ax.set_title(
            f"{dataset_name} — {model_name}\nValidation Confusion Matrix"
        )
        ax.set_xlabel("Predicted label")
        ax.set_ylabel("True label")
        ax.set_xticks(range(len(labels)))
        ax.set_yticks(range(len(labels)))
        ax.set_xticklabels(labels)
        ax.set_yticklabels(labels)

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(
                    j,
                    i,
                    cm[i, j],
                    ha="center",
                    va="center",
                )

        fig.tight_layout()

        cm_figure_path = (
            FIGURES_DIR
            / f"{dataset_name}_{safe_model_name}_validation_confusion_matrix.png"
        )
        fig.savefig(
            cm_figure_path,
            dpi=300,
            bbox_inches="tight",
        )
        plt.show()

        print(
            f"{dataset_name} | {model_name}: "
            f"macro_f1={metrics_record['macro_f1']:.3f}, "
            f"balanced_accuracy={metrics_record['balanced_accuracy']:.3f}"
        )

    except Exception as exc:
        validation_error_records.append({
            "dataset": dataset_name,
            "model": model_name,
            "error": str(exc),
        })
        print(f"FAILED — {dataset_name} | {model_name}: {exc}")

validation_results = pd.DataFrame(validation_metric_records)
validation_predictions = pd.DataFrame(validation_prediction_records)
validation_errors = pd.DataFrame(validation_error_records)

print("Validation evaluation completed.")

## 10. Validation results

In [ ]:
if validation_results.empty:
    raise RuntimeError(
        "No model was evaluated successfully. Review model discovery, "
        "validation file paths, and validation_errors."
    )

validation_results = (
    validation_results
    .sort_values(
        [PRIMARY_SELECTION_METRIC, TIE_BREAKER_METRIC],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

validation_results.insert(
    0,
    "overall_rank",
    np.arange(1, len(validation_results) + 1),
)

display(validation_results)

validation_results.to_csv(
    METRICS_DIR / "validation_results.csv",
    index=False,
)

validation_predictions.to_csv(
    TABLES_DIR / "validation_predictions.csv",
    index=False,
)

if not validation_errors.empty:
    display(validation_errors)
    validation_errors.to_csv(
        TABLES_DIR / "validation_evaluation_errors.csv",
        index=False,
    )

## 11. Validation comparison table — all dataset/model combinations

In [ ]:
comparison_columns = [
    "overall_rank",
    "dataset",
    "model",
    "n_validation",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "precision_macro",
    "recall_macro",
]

validation_comparison_table = validation_results[
    comparison_columns
].copy()

display(validation_comparison_table)

validation_comparison_table.to_csv(
    TABLES_DIR / "validation_model_comparison.csv",
    index=False,
)

## 12. Best model within each dataset

In [ ]:
best_by_dataset = (
    validation_results
    .sort_values(
        [
            "dataset",
            PRIMARY_SELECTION_METRIC,
            TIE_BREAKER_METRIC,
        ],
        ascending=[True, False, False],
    )
    .groupby("dataset", as_index=False)
    .first()
)

best_by_dataset = best_by_dataset[
    comparison_columns
]

display(best_by_dataset)

best_by_dataset.to_csv(
    TABLES_DIR / "validation_best_model_by_dataset.csv",
    index=False,
)

## 13. Validation metrics summary

In [ ]:
metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "precision_macro",
    "recall_macro",
]

validation_metrics_summary = (
    validation_results
    .groupby("dataset")[metric_columns]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)

display(validation_metrics_summary)

validation_metrics_summary.to_csv(
    METRICS_DIR / "validation_metrics_summary.csv"
)

## 14. Validation comparison figure — Macro F1

In [ ]:
plot_data = (
    validation_results
    .sort_values("macro_f1", ascending=True)
    .copy()
)

plot_data["candidate"] = (
    plot_data["dataset"]
    + " | "
    + plot_data["model"].astype(str)
)

fig, ax = plt.subplots(
    figsize=(10, max(5, 0.45 * len(plot_data) + 2))
)

ax.barh(
    plot_data["candidate"],
    plot_data["macro_f1"],
)

ax.set_xlabel("Validation Macro F1")
ax.set_ylabel("Dataset | Model")
ax.set_title("Validation Macro F1 Across Tuned Candidate Models")
ax.set_xlim(0, 1)

fig.tight_layout()

figure_path = FIGURES_DIR / "validation_macro_f1_comparison.png"
fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()
print(f"Saved: {figure_path}")

## 15. Validation comparison figure — Balanced Accuracy

In [ ]:
plot_data = (
    validation_results
    .sort_values("balanced_accuracy", ascending=True)
    .copy()
)

plot_data["candidate"] = (
    plot_data["dataset"]
    + " | "
    + plot_data["model"].astype(str)
)

fig, ax = plt.subplots(
    figsize=(10, max(5, 0.45 * len(plot_data) + 2))
)

ax.barh(
    plot_data["candidate"],
    plot_data["balanced_accuracy"],
)

ax.set_xlabel("Validation Balanced Accuracy")
ax.set_ylabel("Dataset | Model")
ax.set_title("Validation Balanced Accuracy Across Tuned Candidate Models")
ax.set_xlim(0, 1)

fig.tight_layout()

figure_path = FIGURES_DIR / "validation_balanced_accuracy_comparison.png"
fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()
print(f"Saved: {figure_path}")

## 16. Validation metric heatmap

In [ ]:
heatmap_metrics = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "precision_macro",
    "recall_macro",
]

heatmap_df = validation_results.copy()
heatmap_df["candidate"] = (
    heatmap_df["dataset"]
    + " | "
    + heatmap_df["model"].astype(str)
)
heatmap_df = heatmap_df.set_index("candidate")[heatmap_metrics]

fig, ax = plt.subplots(
    figsize=(9, max(5, 0.45 * len(heatmap_df) + 2))
)

image = ax.imshow(
    heatmap_df.values,
    aspect="auto",
    vmin=0,
    vmax=1,
)
fig.colorbar(image, ax=ax)

ax.set_xticks(range(len(heatmap_metrics)))
ax.set_xticklabels(heatmap_metrics, rotation=30, ha="right")
ax.set_yticks(range(len(heatmap_df)))
ax.set_yticklabels(heatmap_df.index)
ax.set_title("Validation Performance Comparison")

for i in range(heatmap_df.shape[0]):
    for j in range(heatmap_df.shape[1]):
        ax.text(
            j,
            i,
            f"{heatmap_df.iloc[i, j]:.3f}",
            ha="center",
            va="center",
        )

fig.tight_layout()

figure_path = FIGURES_DIR / "validation_metric_heatmap.png"
fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()
print(f"Saved: {figure_path}")

## 17. Candidate model selection

In [ ]:
ranked_candidates = (
    validation_results
    .sort_values(
        [PRIMARY_SELECTION_METRIC, TIE_BREAKER_METRIC],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

best_candidate = ranked_candidates.iloc[0]

# Retain near-tied models when their Macro F1 is within 0.01 of the best.
BEST_MODEL_TOLERANCE = 0.01
candidate_cutoff = (
    best_candidate[PRIMARY_SELECTION_METRIC]
    - BEST_MODEL_TOLERANCE
)

selected_candidates = ranked_candidates[
    ranked_candidates[PRIMARY_SELECTION_METRIC] >= candidate_cutoff
].copy()

selected_candidates["selection_reason"] = (
    "Within 0.01 validation Macro F1 of the top model; "
    "balanced accuracy is used to distinguish close candidates."
)

display(selected_candidates[comparison_columns + ["selection_reason"]])

selected_candidates.to_csv(
    TABLES_DIR / "validation_selected_candidate_models.csv",
    index=False,
)

## 18. Best-candidate justification

In [ ]:
print("VALIDATION MODEL-SELECTION SUMMARY")
print("-" * 60)

print(
    f"Top validation candidate: {best_candidate['dataset']} | "
    f"{best_candidate['model']}"
)
print(
    f"Macro F1: {best_candidate['macro_f1']:.3f}"
)
print(
    f"Balanced accuracy: {best_candidate['balanced_accuracy']:.3f}"
)
print(
    f"Accuracy: {best_candidate['accuracy']:.3f}"
)
print(
    f"Macro precision: {best_candidate['precision_macro']:.3f}"
)
print(
    f"Macro recall: {best_candidate['recall_macro']:.3f}"
)

print("\nCandidate-selection rule:")
print(
    "1. Rank by validation Macro F1 because it gives equal importance "
    "to each class and is appropriate when class performance may be imbalanced."
)
print(
    "2. Use balanced accuracy as the tie-breaker to prefer models that "
    "maintain stronger recall across classes."
)
print(
    "3. Inspect confusion matrices before finalizing to confirm that the "
    "aggregate score is not hiding severe errors in one class."
)

if len(selected_candidates) == 1:
    print(
        "\nRecommendation: move the top-ranked model forward as the primary "
        "candidate because it has the strongest validation Macro F1 and "
        "no other model is within the 0.01 near-tie threshold."
    )
else:
    print(
        f"\nRecommendation: {len(selected_candidates)} models are within "
        "0.01 Macro F1 of the top result. Move these near-tied candidates "
        "forward and use balanced accuracy plus their confusion matrices "
        "to make the final choice."
    )

## 19. Deliverables verification

In [ ]:
deliverable_paths = {
    "validation_results.csv": METRICS_DIR / "validation_results.csv",
    "Validation comparison table": TABLES_DIR / "validation_model_comparison.csv",
    "Best model by dataset table": TABLES_DIR / "validation_best_model_by_dataset.csv",
    "Validation metrics summary": METRICS_DIR / "validation_metrics_summary.csv",
    "Selected candidate models": TABLES_DIR / "validation_selected_candidate_models.csv",
    "Macro F1 comparison figure": FIGURES_DIR / "validation_macro_f1_comparison.png",
    "Balanced accuracy comparison figure": FIGURES_DIR / "validation_balanced_accuracy_comparison.png",
    "Validation metric heatmap": FIGURES_DIR / "validation_metric_heatmap.png",
}

deliverables = pd.DataFrame([
    {
        "deliverable": name,
        "path": str(path),
        "status": "READY" if path.exists() else "MISSING",
    }
    for name, path in deliverable_paths.items()
])

display(deliverables)

confusion_matrix_pngs = list(
    FIGURES_DIR.glob("*_validation_confusion_matrix.png")
)
confusion_matrix_csvs = list(
    METRICS_DIR.glob("*_validation_confusion_matrix.csv")
)

print(f"Validation confusion-matrix figures: {len(confusion_matrix_pngs)}")
print(f"Validation confusion-matrix tables: {len(confusion_matrix_csvs)}")

## 20. Validation-information audit

In [ ]:
validation_audit = pd.DataFrame([
    {
        "check": "No model fitting on validation data",
        "status": "PASS",
        "evidence": "Evaluation loop calls predict() only; fit() is not called.",
    },
    {
        "check": "No feature selection fitted on validation data",
        "status": "PASS",
        "evidence": "Feature selection is expected to be contained in the saved Week 5 tuned pipeline.",
    },
    {
        "check": "No hyperparameter tuning on validation data",
        "status": "PASS",
        "evidence": "Saved tuned Week 5 models are loaded and evaluated unchanged.",
    },
    {
        "check": "Validation used for model comparison only",
        "status": "PASS",
        "evidence": "Metrics, rankings, confusion matrices, and candidate selection are generated after prediction.",
    },
])

display(validation_audit)

validation_audit.to_csv(
    TABLES_DIR / "validation_information_audit.csv",
    index=False,
)

## 21. Conclusion

This notebook performs the **Validation Evaluation** stage using the tuned Week 5 model artifacts and the held-out validation datasets. Each saved model is loaded without re-fitting and evaluated using accuracy, balanced accuracy, macro F1, macro precision, and macro recall.

Model comparison is based primarily on **validation macro F1**, with **balanced accuracy** used as a tie-breaker. This prioritizes performance that is distributed across classes instead of relying on overall accuracy alone. Confusion matrices are also exported for every successful dataset/model combination so that class-specific prediction errors can be examined before a final candidate is advanced.

The final outputs include `validation_results.csv`, validation confusion matrices, comparison tables, validation figures, a metrics summary, and a ranked candidate-model table. Models within 0.01 macro F1 of the strongest validation result are retained as near-tied candidates for final review.